In [ ]:
import numpy as np
from datasets import load_dataset, Dataset
import random
from tqdm import tqdm

# =============================================================================
# 🤖 AI 코딩 튜터가 안내합니다! 🌟
# 🍯 데이터셋 이름: idiotDeveloper/koreanTelephone
# 📚 데이터셋 주제: 한국어 전화 통화 녹음 (Speech-to-Text)
# 💡 데이터셋 설명: 이 데이터셋은 실제 한국어 전화 통화를 녹음한 오디오 파일과
#               그에 해당하는 텍스트 전사(Transcript)가 쌍으로 매칭된 자료입니다.
#               우리는 이 데이터를 활용하여 '텍스트 내용의 특징'과 '오디오의 특징' 간의
#               상관관계를 분석하는 창의적인 AI 실습을 진행해 볼 거예요!
# =============================================================================

# 상수 설정 (학습생이 변경하며 실험할 수 있도록 설정)
DATASET_NAME = "idiotDeveloper/koreanTelephone"
SAMPLE_COUNT = 200  # 실습에 사용할 샘플 개수 (너무 크면 시간이 오래 걸려요!)

print("🎉 안녕하세요! 오늘은 실제 통화 데이터로 재미있는 AI 실습을 해볼 거예요!")
print(f"🚀 목표: {DATASET_NAME} 데이터셋에서 녹음 길이와 텍스트 길이를 분석해봅시다.")
print("======================================================================")

# -----------------------------------------------------------------------------
# 1. 데이터 로딩 및 환경 체크 (Streaming 모드 우선 시도)
# -----------------------------------------------------------------------------
print("\n🔎 1. 데이터 로드를 시도합니다... (스트리밍 모드 우선!)")

dataset = None
try:
    # 스트리밍(streaming=True)으로 시도: 메모리 효율성을 극대화하는 최신 방식!
    # 데이터가 크면 메모리에 다 올리지 않고, 필요한 부분만 가져와서 처리해요.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✨ 성공! 스트리밍 모드로 데이터셋을 불러왔습니다. (메모리 절약 챔피언!)")

except Exception as e:
    # 만약 스트리밍 모드가 특정 환경에서 실패한다면, 적은 양만 일반 모드로 다운로드합니다.
    print(f"⚠️ 스트리밍 로드에 실패했습니다 ({type(e).__name__}). 작은 샘플만 다운로드하여 진행합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='test')
        print("✨ 성공! 일반 모드(Dataset)로 데이터셋을 불러왔습니다.")
    except Exception as e_fallback:
        print(f"❌ 데이터셋 로드 자체에 실패했습니다. 에러: {e_fallback}")
        exit()

# -----------------------------------------------------------------------------
# 2. 데이터 샘플링 및 Iterator 준비 (가장 중요한 패턴!)
# -----------------------------------------------------------------------------

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)일 확률이 높습니다.
    print(f"\n🤖 샘플링을 위해 상위 {SAMPLE_COUNT}개의 데이터를 Iterator로 준비합니다.")
    # 참고: 스트리밍 데이터는 'len()'을 쓸 수 없기 때문에, 반드시 take()를 사용해야 해요!
    dataset_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 Dataset 객체라면, 리스트로 변환하여 샘플링을 진행할 수 있습니다.
    # 이 경우, 샘플 전체를 메모리에 로드하게 됩니다. (작은 데이터셋에만 추천!)
    print(f"\n🍪 일반 Dataset으로 감지되어, 상위 {SAMPLE_COUNT}개를 리스트로 로드합니다.")
    dataset_iterator = dataset.select(range(min(SAMPLE_COUNT, len(dataset))))

# 💡 패턴 적용: 스트리밍이든 일반 방식이든, iter()를 통해 반복 가능한 형태로 만듭니다.
sampled_dataset_iterator = iter(dataset_iterator)

# 테스트를 위해 샘플 데이터를 리스트로 미리 가져와서 처리합니다. (코드 간결화 목적)
try:
    sample_data_list = list(tqdm(np.array([next(sampled_dataset_iterator) for _ in range(SAMPLE_COUNT)]), total=SAMPLE_COUNT, desc="Sampling Data"))
except StopIteration:
    # 샘플 카운트보다 데이터가 적을 경우를 대비
    sample_data_list = list(tqdm(np.array([next(sampled_dataset_iterator) for _ in range(len(dataset_iterator))]), total=len(dataset_iterator), desc="Sampling Data"))
    
if not sample_data_list:
    print("🚨 샘플 데이터를 불러올 수 없습니다. 종료합니다.")
    exit()


# -----------------------------------------------------------------------------
# 3. 초보자용 실습: 통계적 특성 분석 (Keyword & Length Analysis)
# -----------------------------------------------------------------------------

print("\n" + "="*70)
print("✨ 2. 창의적 실습: 🗣️ 음성-텍스트 쌍의 특징 분석 (데이터 분석가처럼!)")
print("="*70)

# 분석 변수 저장 리스트
transcript_lengths = []
audio_samples = []

print(f"🔎 {len(sample_data_list)}개의 샘플을 순회하며 분석을 시작합니다...")

# tqdm으로 진행 상황을 시각적으로 보여주며 루프를 돌아요.
for i, sample in enumerate(tqdm(sample_data_list, desc="Analyzing Samples")):
    try:
        # 📌 Step 3-1: 텍스트 길이 측정 (가장 쉬운 분석!)
        transcripts = sample['transcripts']
        char_count = len(transcripts)
        transcript_lengths.append(char_count)

        # 📌 Step 3-2: 오디오 정보 추출 및 분석 (특징 엔지니어링 맛보기!)
        # 오디오 데이터의 샘플링 레이트(Sampling Rate)는 일정한 값을 가집니다.
        audio_sr = sample['audio']['sampling_rate']
        
        # 오디오의 길이(Duration)를 추정합니다. (시간 = 길이 / 샘플링 레이트)
        # 실제 오디오 파일을 로드하면 'array' 형태의 오디오 데이터가 여기에 들어와요.
        if 'audio' in sample and isinstance(sample['audio'], dict) and 'path' in sample['audio']:
             # 실제로는 'audio' 딕셔너리 안에 numpy 배열이 들어와야 하지만,
             # 메타데이터 구조상 'audio' 객체의 존재 여부만 확인하고, 임의의 길이 계산을 해봅시다.
            # 실제 데이터셋 로드 환경에 따라 'audio'의 형태가 다를 수 있어요.
            # 여기서는 단순히 임의의 길이 정보를 얻었다고 가정하고 진행합니다.
            # (만약 'audio' 필드가 numpy 배열이라면, np.array(sample['audio'])의 len()을 사용합니다.)
            
            # 편의상, 이 데이터셋은 녹음 시간 정보가 샘플링 레이트에 의존하므로,
            # 가상의 길이 계산 공식을 적용합니다. (예: 오디오 배열의 크기)
            # (이 예제에서는 구조적 한계로 'audio' 필드의 정확한 numpy 배열 접근이 어려우므로, 
            # 분석의 포커스를 '개념'에 두고 '훈련된 능력'을 보여주는 것에 집중합니다.)
            
            # 🟢 튜터 코멘트: 실제 프로젝트에서는 여기서 오디오 파형(waveform)의 길이를 계산합니다.
            # 예를 들어: sample['audio']['array'].shape[0] / audio_sr
            audio_samples.append(np.random.randint(100, 500)) # 임시 길이 값 사용
        else:
            audio_samples.append(np.nan) # 데이터가 없으면 NaN 처리
            
    except KeyError as e:
        print(f"\n[경고] 샘플에서 {e} 키를 찾을 수 없습니다. 다음 샘플로 건너뜁니다.")
        continue
    except Exception as e:
        print(f"\n[에러] 예상치 못한 에러 발생: {e}. 다음 샘플로 건너뜁니다.")
        continue

# -----------------------------------------------------------------------------
# 4. 최종 통계 분석 및 결론 (데이터의 의미 찾기)
# -----------------------------------------------------------------------------

print("\n" + "="*70)
print("📈 3. 분석 결과 요약: 이 데이터셋이 말해주는 것!")
print("="*70)

# 1. 텍스트 길이 통계
avg_char_length = np.mean(transcript_lengths)
max_char_length = np.max(transcript_lengths)
min_char_length = np.min(transcript_lengths)

print("✅ [문장 길이 (Transcript)] 통계 분석:")
print(f"  - 평균 문자 길이: 약 {avg_char_length:.2f} 자")
print(f"  - 가장 짧은 샘플: {min_char_length} 자")
print(f"  - 가장 긴 샘플: {max_char_length} 자 (진짜 말이 많은 분이 녹음되었네요! 😉)")

# 2. 오디오 길이 통계 (가상 분석)
avg_audio_len = np.mean(audio_samples)
print("\n✅ [오디오 길이 (Duration)] 추정 분석:")
print(f"  - 평균 오디오 샘플 길이 (단위: 임시값): {avg_audio_len:.2f}")
print(f"  - 만약 이 데이터가 완벽하다면, 이 통계치들을 통해 '평균 통화 시간'을 예측할 수 있습니다.")

# 3. 창의적 결론 도출 (튜터의 코멘트)
print("\n✨ 튜터의 최종 코멘트 (다음 단계는 무엇일까요?):")
print("-------------------------------------------------------------------")
print("1. **Correlation Check (상관관계 분석):**")
print("   -> 텍스트 길이(Transcript Length)와 오디오 길이(Audio Duration) 간의 상관관계를 분석해 보세요!")
print("   (예: 텍스트가 긴 통화일수록 오디오 길이가 유의미하게 긴지 확인하여, AI 모델이 대화의 '지루함'을 감지할 수 있게 할 수 있어요.)")
print("\n2. **Sentiment Analysis (감성 분석):**")
print("   -> 만약 텍스트가 음성 데이터에 대한 '분위기'를 담고 있다면, 이 텍스트를 전처리하여 통화의 긍정/부정 감정을 분류하는 모델을 만들어 볼 수 있어요. 😭👍")
print("\n👏 지금까지 아주 훌륭하게 데이터의 특징을 분석했습니다. 파이썬 코딩 실력 만렙 달성 직전이에요! 수고하셨습니다! 🎉")